In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import os

import tqdm
import glob

In [ ]:
import torch
from torch.utils.data import DataLoader

from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import r2_score
from sktime.performance_metrics.forecasting import (
    MeanAbsoluteScaledError, 
    MeanAbsolutePercentageError, 
    MeanSquaredError, 
    MeanAbsoluteError
)

In [3]:
import helper as hl
import models as ml
import dataset as ds

# Define Global Variables

In [4]:
SR_FREQ = '12H'
device = torch.device('cpu')

In [5]:
cli_args = dict(
    # LSTM PARAMS
    data='paloalto',    # dundee | porto | boulder | paloalto
    rnn_cell='gru',   # lstm   | gru
    bi=True,
    hidden_size=24, 
    num_layers=1, 
    fc_layers='24', # D3.2
    # DATA PARAMS
    sr_freq=SR_FREQ, 
    min_pts=100,
    # TRAINING PARAMS
    bs=32, 
    length=48, 
    stride=1, 
    n_rounds=50,
)
cli_args['num_clients'] = 8 if cli_args['data'] == 'dundee' else 4

windowing_params = dict(
    length_min=cli_args['min_pts'], 
    length_max=cli_args['length'], 
    stride=cli_args['stride'],
    rnn_feats=[
        'day_sin', 'day_cos',
        'hour_sin', 'hour_cos', 
        'week_sin', 'week_cos', 
        #
        'power_curr_logdelta',
        'power_next_step1_extrap',
        # 
        'power_curr_std', 
        'power_curr_ema',
        #
        f'downtime_scaled',
        #
        'no_of_sessions_scaled',
        'charging_time',
        'power_curr', 
    ],
    # Extra features to include in the Fully Connected (FC) layer (excl. ```building``` and ```model```)
    fc_feats=[
        'power_output_kW', 
    ],
    y_feats=['power_next'], 
)

# Load Dataset

In [6]:
df_evse_demand_v3 = pd.read_pickle(
    os.path.join(
        '../data', 'pkl', 
        f"{cli_args['data']}_data.demand_{cli_args['sr_freq']}_{cli_args['min_pts']}_points.enriched.v4.pickle"
    )
).sort_index()

df_evse_demand_v3.reset_index(level=2, inplace=True)
df_evse_demand_v3.timestamp = df_evse_demand_v3.timestamp.dt.tz_localize(None)
df_evse_demand_v3.set_index('timestamp', append=True, inplace=True)

In [7]:
df_tr_dev_test = pd.concat(
    {
        cluster_id: pd.read_pickle(
            cluster_id_path
        )
        # 
        for cluster_id, cluster_id_path in enumerate(
            sorted(
                glob.glob(
                    f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}_sequences_{len(windowing_params["rnn_feats"])}_rnn_inputs_{len(windowing_params["fc_feats"])}_fc_inputs_{len(windowing_params["y_feats"])}_outputs_length_{windowing_params["length_max"]}_stride_{windowing_params["stride"]}.cluster_*.v4.pickle'
                )
            )
        )
    },
    names=['cluster_id']
)

In [8]:
building_tokens, building_token_lookup = (
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_building_tokens_v4.pkl'), 
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_building_token_lookup_v4.pkl')
)
model_tokens, model_token_lookup = (
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_model_tokens_v4.pkl'), 
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_model_token_lookup_v4.pkl')
)

In [9]:
df_evcs_meta = pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data.metadata.v3.pickle')

# Load Model

In [10]:
building_embeddings = torch.nn.Embedding(len(building_token_lookup), 15) 
model_embeddings = torch.nn.Embedding(len(model_token_lookup), 3) 

In [11]:
model_params = dict(
    location_embeddings=building_embeddings,
    model_embeddings=model_embeddings,
    input_size=len(windowing_params['rnn_feats']),
    scale=None,
    rnn_cell=getattr(torch.nn, cli_args['rnn_cell'].upper()),
    bidirectional=cli_args['bi'],
    num_layers=cli_args['num_layers'],
    hidden_size=cli_args['hidden_size'],
    misc_size=len(windowing_params['fc_feats']),
    fc_layers=[int(i) for i in cli_args['fc_layers'].split(',')],
    output_size=len(windowing_params['y_feats']),
)

In [12]:
fededf_heavy_dirs = {
    'dundee_lstm': 'fededf_ver2025-08-30_13-35-16_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_bilstm': 'fededf_ver2025-08-30_13-34-38_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_gru': 'fededf_ver2025-08-30_19-24-16_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_bigru': 'fededf_ver2025-08-31_04-10-08_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    # 
    'porto_lstm': 'fededf_ver2025-08-31_04-11-57_porto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'porto_bilstm': 'fededf_ver2025-08-31_08-08-53_porto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'porto_gru': 'fededf_ver2025-08-31_07-04-20_porto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'porto_bigru': 'fededf_ver2025-08-31_10-41-12_porto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    # 
    'boulder_lstm': 'fededf_ver2025-10-11_17-03-19_boulder_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'boulder_bilstm': 'fededf_ver2025-10-11_22-31-59_boulder_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'boulder_gru': 'fededf_ver2025-10-12_06-29-26_boulder_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'boulder_bigru': 'fededf_ver2025-10-12_06-30-59_boulder_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    # 
    'paloalto_lstm': 'fededf_ver2025-10-12_07-45-54_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'paloalto_bilstm': 'fededf_ver2025-10-12_07-46-42_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'paloalto_gru': 'fededf_ver2025-10-12_12-00-18_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'paloalto_bigru': 'fededf_ver2025-10-13_15-28-16_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
}

fededf_light_dirs = {
    'dundee_lstm': 'fededf_ver2025-11-30_09-46-38_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_bilstm': 'fededf_ver2025-11-30_11-08-24_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_gru': 'fededf_ver2025-11-30_11-08-33_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_bigru': 'fededf_ver2025-11-30_12-07-02_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
}

fededf_dirs = fededf_heavy_dirs # fededf_heavy_dirs | fededf_light_dirs

In [13]:
MODEL_RNN_TYPE_NAME = f'{"bi" if model_params["bidirectional"] else ""}{cli_args["rnn_cell"].upper()}'
                    
save_path_best = os.path.join(
    '..', 
    'data', 
    'pth', 
    fededf_dirs[f'{cli_args["data"]}_{"bi" if model_params["bidirectional"] else ""}{cli_args["rnn_cell"]}'],
    f'fededf_{cli_args["data"]}.flwr_global.epoch{cli_args["n_rounds"]}.pth' 
)

print(save_path_best)

../data/pth/fededf_ver2025-10-13_15-28-16_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1/fededf_paloalto.flwr_global.epoch50.pth


In [14]:
def load_model(path, model_config, device):
    # # Evaluate Best Model
    checkpoint = torch.load(path, map_location=device)

    fededf_model = ml.EnergyDemandForecasting_v2(
        **model_config
    )
    fededf_model.to(device)

    # Assign each NumPy array to the corresponding layer in the model
    with torch.no_grad():  # Disable gradient tracking to avoid issues during assignment
        for param, np_array in zip(fededf_model.parameters(), checkpoint['parameters']):
            # Convert NumPy array to a torch tensor with the same dtype as model parameters
            param.copy_(torch.tensor(np_array, dtype=param.dtype))

    fededf_model.eval()
    return fededf_model

# Make Predictions

In [19]:
def fededf_model_inference(model, data_loader):
    y_true_oid_nn, y_pred_oid_nn = [], []

    with torch.no_grad():
        for xb, yb, lb, *args in (pbar := tqdm.tqdm(data_loader, leave=False, total=len(data_loader), dynamic_ncols=True)):
            # print(f'{xb.shape=}\t {yb.shape=}\t {lb.shape=}')
            xb, yb = xb.to(device), yb.to(device)    # Model Inference
            args = (arg.to(device) for arg in args)
            y_pred = model(xb.float(), lb, *args).detach()
            
            y_true_oid_nn.append(yb)
            y_pred_oid_nn.append(y_pred)

    y_true_oid_nn, y_pred_oid_nn = np.concatenate(y_true_oid_nn), np.concatenate(y_pred_oid_nn)
    return y_true_oid_nn, y_pred_oid_nn

In [20]:
def fededf_model_inference_table(evcs_dataset_windows_test, df_evcs_meta, y_true_oid_nn, y_pred_oid_nn, t_horizon=0):
    evcs_dataset_windows_y_pred_time_axis = evcs_dataset_windows_test.time_axes.apply(lambda l: l[-1]).values

    lstm_results = pd.concat(
        {
            MODEL_RNN_TYPE_NAME:pd.Series(
                y_pred_oid_nn[:, t_horizon, :].squeeze(),
                index=[
                    evcs_dataset_windows_test.index.get_level_values(0),
                    evcs_dataset_windows_y_pred_time_axis
                ],
            ).rename_axis(['oid', 'timestamp']),
            'power_next':pd.Series(
                y_true_oid_nn[:, t_horizon, :].squeeze(),
                index=[
                    evcs_dataset_windows_test.index.get_level_values(0),
                    evcs_dataset_windows_y_pred_time_axis
                ]
            ).rename_axis(['oid', 'timestamp']),
        }, 
        axis=1
    )

    lstm_results = lstm_results.loc[~lstm_results.index.duplicated(keep='last')].copy() # Drop duplicated entries (in case of overlapping windows)
    lstm_results = lstm_results.unstack('timestamp')

    return lstm_results.groupby('oid', group_keys=False).apply(
        lambda l: l * (df_evcs_meta.loc[l.name, 'power_output_kW'] * (pd.Timedelta(SR_FREQ).total_seconds() / 3600))
    )

In [18]:
identity_function = FunctionTransformer(None) # We do not need a scaler for now, so we use the Identity function...

evcs_dataset_windows_test = df_tr_dev_test.reorder_levels([1,2,0]).xs(3, level=1, drop_level=False).dropna().copy()

test_dataset = ds.EDFDataset_v2(building_tokens, model_tokens, building_token_lookup, model_token_lookup, evcs_dataset_windows_test, scaler=identity_function)
test_loader = DataLoader(test_dataset,  batch_size=1, shuffle=False, collate_fn=test_dataset.pad_collate)

In [21]:
fededf_global_epoch_i = load_model(
    save_path_best,
    model_params,
    device
)

y_true_oid_nn, y_pred_oid_nn = fededf_model_inference(fededf_global_epoch_i, test_loader)
edf_results = fededf_model_inference_table(evcs_dataset_windows_test, df_evcs_meta, y_true_oid_nn, y_pred_oid_nn, t_horizon=0)

/Users/andrewt/Documents/DataStories-UniPi/FedEDF/src/models.py:44: UserWarning: Instantiated instance without standardization. Falling back to identity function...
  warnings.warn("Instantiated instance without standardization. Falling back to identity function...")
/var/folders/p6/kw8t6dgj12v1289kljz8mz4w0000gn/T/ipykernel_25422/3688296761.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  lambda l: l * (df_evcs_meta.loc[l.name, 'power_output_kW'] * (pd.Timedelta(SR_FREQ).total_seconds() / 3600))


In [22]:
edf_results_final = edf_results.clip(lower=0).stack(level=1, future_stack=True).dropna()
edf_results_final

biGRU  power_next
oid                       timestamp                                 
PALO ALTO CA / BRYANT #1  2020-02-22 12:00:00   8.234468    0.000000
                          2020-02-23 00:00:00  16.743965    0.000000
                          2020-02-23 12:00:00   8.771297    4.468506
                          2020-02-24 00:00:00  16.322033    9.622494
                          2020-02-24 12:00:00  10.528006    2.133044
...                                                  ...         ...
PALO ALTO CA / WEBSTER #3 2020-12-29 00:00:00  12.134576   37.089767
                          2020-12-29 12:00:00  10.967976   21.067846
                          2020-12-30 00:00:00  19.829243   26.638155
                          2020-12-30 12:00:00  10.778436    0.000000
                          2020-12-31 00:00:00  19.823671    7.075001

[13300 rows x 2 columns]

# Calculate metrics

In [23]:
train_set_lookup = (
    df_tr_dev_test
    .xs(1, level=2)
    .time_axes
    .explode()
    .groupby(level=1)
    .max()
    .apply(
        lambda l: str(
            l.date()
        )
    )
)

available_oids = list(
    set(
        df_tr_dev_test.xs(3, level=2).dropna().index.get_level_values(1).unique()

    ).intersection(
        set(
            df_tr_dev_test.xs(1, level=2).dropna().index.get_level_values(1).unique()
        )
    )
)

df_evse_demand_v3_train_set = df_evse_demand_v3.loc[
    pd.IndexSlice[
        :, 
        available_oids,
        :
    ]
].groupby(
    'oid', 
    group_keys=False
).apply(
    lambda l: l.loc[
        pd.IndexSlice[
            :, 
            :, 
            :train_set_lookup.loc[l.name]
        ]
    ]
).copy()

y_train = df_evse_demand_v3_train_set['power_curr']
oid_indices = df_evse_demand_v3_train_set['power_curr'].sort_index().groupby('oid', observed=False).groups

In [24]:
model_results_metrics = hl.evaluate_predictions(
    edf_results_final.dropna(),
    y_true_name='power_next',
    y_pred_names=[MODEL_RNN_TYPE_NAME],
    eval_funs=[
        ('MASE_pct', MeanAbsoluteScaledError(sp=24), {'y_train':y_train, 'oid_indices':oid_indices}),
        ('SMAPE_pct', MeanAbsolutePercentageError(symmetric=True), {}),
        ('MAAPE_rads', hl.mean_arctangent_absolute_percentage_error, {}),
        ('WAPE_pct', hl.wape, {}),
        ('RMSE_kW', MeanSquaredError(square_root=True), {}),
        ('MAE_kW', MeanAbsoluteError(), {}),
        ('R2', r2_score, {})
    ]
)

In [25]:
model_results_metrics.groupby(level=0, sort=False).describe().T.loc[
    pd.IndexSlice[:, ['mean', '25%', '50%', '75%']], :
]

0_MASE_pct  1_SMAPE_pct  2_MAAPE_rads  3_WAPE_pct  4_RMSE_kW  \
biGRU mean    0.720087     1.303239      1.066720    3.157887  10.041538   
      25%     0.531521     1.059862      0.888549    0.938881   9.178849   
      50%     0.700502     1.268877      1.033274    1.211382  10.083893   
      75%     0.879771     1.423243      1.177522    1.682874  12.052746   

            5_MAE_kW      6_R2  
biGRU mean  8.133510 -0.708065  
      25%   6.756902 -0.330075  
      50%   8.159037 -0.175039  
      75%   9.922497  0.088189

In [26]:
edf_results_final.dropna().to_pickle(f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}.EVSE__Fed{MODEL_RNN_TYPE_NAME}__model.v4.pickle')
model_results_metrics.to_pickle(f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}.EVSE__Fed{MODEL_RNN_TYPE_NAME}__model.metrics.v4.pickle')